# ML Cost Optimization: GPU, Spot & Resource Management — Module 3 Architecture

## Learning Objectives
- Optimize GPU utilization and reduce training/inference costs
- Implement spot instance strategies with fallback
- Profile model inference cost vs. latency tradeoff
- Compare cloud provider pricing for ML workloads

## Why This Matters
Module 3 notebook 5 previously only mentioned GPU pricing without implementing cost strategies. This version implements actual cost optimization techniques: spot instance bidding, GPU utilization profiling, mixed precision, and cross-cloud price comparison — essential skills for ML Architects responsible for budget-conscious design.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
sns.set_style("whitegrid")
np.random.seed(42)
%matplotlib inline

# GPU price tracking (simulated cloud pricing data)
pricing_data = pd.DataFrame({
    "provider": ["AWS (p4d.24xlarge)", "AWS (p3.8xlarge)", "GCP (n1-standard-8 with A100)", "Azure (NC6s_v3)"]
})

# Simulated hourly prices (historical average, USD)
pricing_data["hourly_price_usd"] = [3.06, 3.09, 2.85, 2.95]
pricing_data["gpu_type"] = ["NVIDIA A100", "NVIDIA V100", "NVIDIA A100", "NVIDIA V100"]
pricing_data["memory_gb"] = [48, 32, 40, 32]

print("Cloud GPU Pricing Comparison (hourly, average USD)")
print(pricing_data.to_string(index=False))


## 1. Spot Instance Strategy with Fallback
# Spot instances can be 60-90% cheaper but may be reclaimed with short notice.

In [ ]:
# Simulate spot bidding with fallback to on-demand
import random

# Bid as a percentage of on-demand price
def spot_bidding(strategy="conservative"):
    if strategy == "aggressive":
        bid_pct = random.uniform(0.50, 0.70)  # Bid 50-70% of on-demand
    elif strategy == "conservative":
        bid_pct = random.uniform(0.70, 0.85)  # Bid 70-85% of on-demand
    else:  # market-based
        bid_pct = random.uniform(0.85, 0.95)

    return bid_pct

# Cost comparison: on-demand vs. spot strategies
on_demand_price = 3.06  # USD/hour for A100
n_hours = 100

results = {}
for strategy in ["aggressive", "conservative", "market"]:
    total_cost = 0
    interruptions = 0
    for _ in range(n_hours):
        bid = spot_bidding(strategy)
        # Simulate spot price (always between 0.5x and 1.0x on-demand)
        spot_price = on_demand_price * random.uniform(0.5, 1.0)
        
        if bid >= spot_price:
            # Use spot instance
            total_cost += spot_price
        else:
            # Fallback to on-demand
            total_cost += on_demand_price
        
        # Random interruption (simulated): 5-15% of spot hours
        if random.random() < 0.10:
            interruptions += 1

    results[strategy] = {
        "total_cost": total_cost,
        "avg_hourly": total_cost / n_hours,
        "interruptions": interruptions,
        "savings_vs_ondemand": (on_demand_price * n_hours - total_cost) / (on_demand_price * n_hours) * 100
    }

print("\nSpot instance cost comparison (100 hours, A100 $3.06/hr on-demand):")
for strategy, result in results.items():
    print(f"  {strategy:12s}: ${result['total_cost']:.2f} total, ${result['avg_hourly']:.2f}/hr, "
          f"{result['interruptions']} interruptions, {result['savings_vs_ondemand']:.1f}% savings")


## 2. GPU Utilization Profiling
# Measure actual GPU utilization to identify over-provisioning opportunities.

In [ ]:
# Profile GPU utilization during training/inference
# Nsight profiling is the standard tool, here we simulate
print("GPU Utilization Profiling Summary:")
print("  • Measure: sm_utilization, memory_copy, compute_occupation")
print("  • Target: >80% utilization for cost-efficient training")
print("  • Low utilization (<50%) indicates opportunity for: larger batch size, model parallelism, or mixed precision")


In [ ]:
# Simulated profiling: batch size vs. GPU utilization
batch_sizes = [16, 32, 64, 128, 256]
utilization = [52, 68, 78, 85, 87]  # % simulation

plt.figure(figsize=(8, 4))
plt.plot(batch_sizes, utilization, 'o-', linewidth=2, markersize=8)
plt.xlabel("Batch Size")
plt.ylabel("GPU Utilization (%)")
plt.title("Batch Size → GPU Utilization Tradeoff")
plt.grid(True, alpha=0.3)
plt.axhline(80, color='red', linestyle='--', alpha=0.5, label="Target 80%")
plt.legend()
plt.show()

# Find optimal batch size (first to exceed 80% target)
optimal_bs = next(bs for bs, util in zip(batch_sizes, utilization) if util >= 80)
print(f"Optimal batch size for ≥80% utilization: {optimal_bs}")


## 3. Mixed Precision Training (AMP)
# Automatic Mixed Precision (AMP) reduces GPU memory & speeds up training.

In [ ]:
# PyTorch Automatic Mixed Precision
import torch.cuda.amp as amp

# Wrap training loop with autocast
scaler = amp.GradScaler()

# Simulated training step with mixed precision
def training_step_amp(model, optimizer, scaler, data, target):
    model.train()
    with amp.autocast():
        output = model(data)
        loss = criterion(output, target)
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()
    return loss.item()

print("Mixed precision training pattern: autocast + GradScaler")


## 4. Cross-Cloud Price Comparison
# Compare on-demand prices across major providers for common GPU types.

In [ ]:
# Extended pricing comparison with more GPU types
all_pricing = pd.DataFrame({
    "provider": ["AWS", "AWS", "AWS", "GCP", "GCP", "Azure", "Azure", "LambdaLabs", "Vast.ai"],
    "gpu_type": ["A100", "V100", "A10", "A100", "A10", "V100", "A10", "A100", "A100"],
    "hourly_price_usd": [3.06, 1.25, 0.40, 2.85, 0.95, 1.50, 0.35, 3.15, 1.80],
    "memory_gb": [48, 32, 24, 40, 24, 32, 24, 40, 24]
})

# Find cheapest A100 and V100
a100_cheapest = all_pricing[all_pricing.gpu_type == 'A100'].sort_values('hourly_price_usd').iloc[0]
v100_cheapest = all_pricing[all_pricing.gpu_type == 'V100'].sort_values('hourly_price_usd').iloc[0]

print("Cheapest GPU options:")
print(f"  Cheapest A100: {a100_cheapest['provider'].values[0]} @ ${a100_cheapest['hourly_price_usd']:.2f}/hr")
print(f"  Cheapest V100: {v100_cheapest['provider'].values[0]} @ ${v100_cheapest['hourly_price_usd']:.2f}/hr")


## 5. Summary

| Concept | Skillsoft Coverage | Notebook Feature |
|---------|-------------------|-----------------|
# GPU price comparison | Module 3 Architecture | ✅ Cross-cloud pricing table +
# Spot instance strategy | Module 3 Architecture | ✅ Bid strategies + interruption simulation |
# GPU utilization profiling | Module 3 Architecture | ✅ Batch size → utilization curve +
# Mixed precision (AMP) | Module 3 Architecture | ✅ PyTorch autocast + GradScaler pattern |
# Cost optimization techniques | Module 3 Architecture | ✅ End-to-end cost reduction strategies |

Key Takeaway: ML Architects must continuously optimize compute costs through spot strategies, batch sizing, mixed precision, and provider selection — often saving 40-60% of ML infrastructure budget.